In [1]:
import my_data_manager as mdm
import pandas as pd
import cluster_utils as cu

cat = mdm.load_cat("Data/galaxies_subhalos_zphot.cat")
halos = mdm.separate_halos(cat)
clusters, groups = mdm.separate_clusters(halos)
clusters = clusters[:678]

/home/gasep/Projects/GitHub/ClusteringComparison/ClusteringComparison/my_data_manager.py:56: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(cat, delim_whitespace=True, header=None)
/home/gasep/Projects/GitHub/ClusteringComparison/ClusteringComparison/my_data_manager.py:56: DtypeWarning: Columns (0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(cat, delim_whitespace=True, header=None)


In [2]:
import astro_utils as au
from astropy.stats import biweight_location, biweight_scale

clusters_mpc = []

for cluster in clusters:
    cluster = cluster.copy()

    cluster["RA"] = pd.to_numeric(cluster["RA"], errors="coerce")
    cluster["DEC"] = pd.to_numeric(cluster["DEC"], errors="coerce")
    cluster["z_app"] = pd.to_numeric(cluster["z_app"], errors="coerce")

    ra_center = biweight_location(cluster["RA"].values.astype(float))
    dec_center = biweight_location(cluster["DEC"].values.astype(float))
    z_center = biweight_location(cluster["z_app"].values.astype(float))

    x, y, z = au.gal_3D_Mpc_coords(
        ra=cluster["RA"].values,
        dec=cluster["DEC"].values,
        redshift_gal=cluster["z_app"].values,
        x_center=ra_center,
        y_center=dec_center,
        z_center=z_center
    )

    cluster["X_Mpc"] = x
    cluster["Y_Mpc"] = y
    cluster["Z_Mpc"] = z

    clusters_mpc.append(cluster)

clusters = clusters_mpc

In [3]:
print(clusters[0].columns)

Index(['galaxyId', 'haloId', 'pos_x', 'pos_y', 'pos_z', 'vel_x', 'vel_y',
       'vel_z', 'redshift', 'snapshot', 'sfr', 'm_star', 'm_coldgas',
       'Z_coldgas', 'R_coldgas', 'sfh_bin', 'sfh_nbin', 'RA', 'DEC', 'z_geo',
       'd_comoving', 'z_app', 'mag_u', 'mag_g', 'mag_r', 'mag_i', 'mag_z',
       'z_phot', 'firstHaloInFOFGroupId', 'log(m_200)', 'X_Mpc', 'Y_Mpc',
       'Z_Mpc'],
      dtype='object', name=0)


In [4]:
clean_clusters = [df[df.groupby("haloId")["haloId"].transform("count") > 3].reset_index(drop=True) for df in clusters]

N = 21

splus_clusters = [df[pd.to_numeric(df['mag_r'], errors = 'coerce') >= N].copy() for df in clusters]

clusters = cu.retag_noise(clusters)
splus_clusters = cu.retag_noise(splus_clusters)

cluster_samples = {
    "Raw": clusters,
    "Clean": clean_clusters,
    "SPLUS": splus_clusters
}

In [5]:
import clustering_methods as clustering
import numpy as np

algorithms = {}


algorithms['BGMM'] = clustering.run_GMM
algorithms['DBSCAN'] = clustering.run_DBSCAN
algorithms['HDBSCAN'] = clustering.run_HDBSCAN

params = {
    "max_clusters" : 0.1,
    "covariance_type" : 'full',
    "min_cluster_size" : 4,
    "min_eps" : 0.5,
    "max_eps" : np.inf,
    "linkage" : 'ward',
    "clustering_threshold" : 0.5,
    "max_iter" : 1000
}

In [7]:
import time
import my_data_manager as mdm

keys = ["Raw","Clean","SPLUS"]
algs = ["HDBSCAN","DBSCAN","BGMM"]

for key in cluster_samples.keys():
    
    samples = cluster_samples[key]

    for alg in algs:

        predictions = []
        iteration_times = [] 

        for sample in samples:
            
                X_data = sample["X_Mpc"].values
                Y_data = sample["Y_Mpc"].values
                Z_data = sample["Z_Mpc"].values
                
                data = np.column_stack((X_data, Y_data))
                data = np.asarray(data, dtype=np.float64)

                start_time = time.perf_counter()
                labels, probs, c = algorithms[alg](data, params)
                end_time = time.perf_counter()
                
                delta_time = end_time - start_time
                # 2. Store the specific time for this iteration
                iteration_times.append(delta_time) 
                
                sample["firstHaloInFOFGroupId"] = (
                    sample["firstHaloInFOFGroupId"]
                    .astype(str)
                )

                fof_id = sample["firstHaloInFOFGroupId"].iloc[0]
                labels = np.insert(labels.astype(str), 0, fof_id)

                predictions.append(labels)
        
        result = []
        result.append((iteration_times, predictions))

        mdm.save_xlsx(result, f"Results/{key}_samples/{alg}_3D")

/home/gasep/Projects/GitHub/ClusteringComparison/.venv/lib/python3.11/site-packages/sklearn/mixture/_base.py:293: ConvergenceWarning: Best performing initialization did not converge. Try different init parameters, or increase max_iter, tol, or check for degenerate data.
  warnings.warn(
